# Context-Deference — driver (multi-model)

`CD_MODEL` picks the target model; every Drive cache is namespaced under `{BASE}/{MODEL_TAG}/`. Set `CD_MODEL` (+ optional `CD_LAYER`) in Setup and run top-to-bottom. Run the one-time migration cell once to relocate the existing flat Qwen caches.

Runs on **Colab** (upload `context-deference.zip`) or **locally in VS Code** from a repo checkout (nothing to upload; set `HF_HOME` to a persistent disk so models download once, and `CD_BASE` for the cache root). `CD_N_CTX=8192` is required for Llama-3.1 (TransformerLens caps it at 2048 -> rotary assert on long kc prompts).


## 0 · Setup

In [ ]:
# === 0 · Setup — Colab (zip upload) OR local VS Code checkout; no zip, no re-download locally ===
import os, sys, subprocess
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    Z = "/content/context-deference.zip"
    assert os.path.exists(Z), "upload context-deference.zip to /content (Files panel) first"
    print(os.path.getsize(Z), "bytes"); os.system("rm -rf /content/suppression"); os.system(f"cd /content && unzip -oq {Z}")
    ROOT = "/content/suppression"
else:
    ROOT = os.environ.get("CD_REPO") or subprocess.run(["git", "rev-parse", "--show-toplevel"],
                                                        capture_output=True, text=True).stdout.strip() or os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
HAS_FIX = "CD_N_CTX" in open("src/model.py").read()
print("IN_COLAB:", IN_COLAB, "| repo root:", ROOT, "| src has the n_ctx fix:", HAS_FIX)
assert HAS_FIX, "this src/model.py predates the n_ctx fix (2026-09-18): pull the repo / re-upload the rebuilt zip"


In [ ]:
if IN_COLAB: os.system("pip -q install -r requirements.txt hf_transfer")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import login, whoami
try:
    print("HF token cached for:", whoami()["name"])
except Exception:
    login()                                   # once; afterwards cached under HF_HOME
print("HF_HOME:", os.environ.get("HF_HOME", "(default ~/.cache/huggingface — locally, point HF_HOME at a persistent disk)"))
print("ready:", os.getcwd())


In [ ]:
import os
os.environ["CD_MODEL"] = "llama-3.1-8b-instruct"   # the second model
os.environ["CD_LAYER"] = "18"                        # depth-matched to Qwen L16 (16*32/28 ≈ 18)
os.environ["CD_N_CTX"] = "8192"                      # REQUIRED for Llama-3.1: TL caps n_ctx at 2048 -> rotary assert on kc
os.environ.setdefault("CD_N_RANDOM", "3")             # random-direction axes (protocol: 3; they do not enter the z-test)
print("CD_MODEL =", os.environ["CD_MODEL"], "| CD_LAYER =", os.environ["CD_LAYER"], "| CD_N_CTX =", os.environ["CD_N_CTX"])

In [ ]:
# === Config + imports ===  (LAYER + MODEL set here; every cache is namespaced by model)
import gc, json
import numpy as np, torch
from src import model as M, data as D, directions as Dir, projection as P, judge as J, universal as U
torch.set_grad_enabled(False)

cfg        = D.load_behaviors_config("configs/behaviors.yaml")
models_cfg = D.load_yaml("configs/models.yaml")
MODEL      = os.environ.get("CD_MODEL", "qwen2.5-7b-instruct")            # <-- set CD_MODEL for a new target model
JUDGE_LLM  = os.environ.get("CD_JUDGE_LLM", "Qwen/Qwen2.5-7B-Instruct")   # judge stays fixed across target models
CONTRAST   = os.environ.get("CD_CONTRAST", cfg["contrast"]["mode"])
SUBSET     = os.environ["CD_SUBSET"].split(",") if os.environ.get("CD_SUBSET") else cfg["mvp_subset"]
mc         = M.resolve_model_cfg(models_cfg, MODEL)
LAYER      = int(os.environ["CD_LAYER"]) if os.environ.get("CD_LAYER") else (mc.get("default_layer") or 16)  # mid-stack
MAXNT      = int(os.environ.get("CD_MAX_NEW_TOKENS", "64"))
POS        = -1
MAXB       = int(os.environ.get("CD_MAX_PAIRS", "0")) or None            # 40 = FAST SMOKE, None = full

try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False
def _mount():                                                            # Drive only exists on Colab
    if IN_COLAB:
        from google.colab import drive; drive.mount("/content/drive")
BASE      = os.environ.get("CD_BASE") or ("/content/drive/MyDrive/context-deference" if IN_COLAB
                                          else os.path.join(os.getcwd(), "results", "cache"))   # cache root
N_RANDOM  = int(os.environ.get("CD_N_RANDOM", "3"))
MODEL_TAG = MODEL.replace("/", "_")                                     # filesystem-safe model id
MDIR      = f"{BASE}/{MODEL_TAG}"                                       # <-- ALL caches for THIS model live here
def _free(): gc.collect(); (torch.cuda.empty_cache() if torch.cuda.is_available() else None)
print("target:", mc["tl_name"], "| judge:", JUDGE_LLM, "| LAYER:", LAYER,
      "| behaviors:", SUBSET, "| max_pairs:", MAXB)
print("cache dir (per-model):", MDIR, "| IN_COLAB:", IN_COLAB)

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # makes CUDA errors synchronous / tracebacks real
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
s   = M.format_chat(bundle, ["What is the capital of France?"])
tok = bundle.model.to_tokens(s, prepend_bos=False)
dv  = bundle.model.cfg.d_vocab
print("d_vocab:", dv, "| token id range:", int(tok.min()), "->", int(tok.max()))
print("OUT OF RANGE:", bool((tok >= dv).any() or (tok < 0).any()))
bad = tok[(tok >= dv) | (tok < 0)]
if bad.numel(): print("offending ids:", sorted(set(bad.tolist())))
print("n_ctx:", bundle.model.cfg.n_ctx, "(Llama-3.1 needs 8192; TL default 2048 asserts on long kc prompts)")
_kc = sorted(D.load_pairs("knowledge_conflict", cfg), key=lambda p: len(p.prompt_manip))[-3:]     # the 3 longest passages
_g = M.generate(bundle, M.format_chat(bundle, [p.prompt_manip for p in _kc]), max_new_tokens=32)
print("longest-kc smoke OK (no rotary assert):", repr(_g[0][:80]))


## 0.5 · One-time cache migration (Qwen flat caches -> qwen2.5-7b-instruct/)

In [ ]:
# === ONE-TIME migration: move existing FLAT caches into the Qwen model folder. ===
# Your existing flat caches are all Qwen's. This relocates them under qwen2.5-7b-instruct/ so the
# new model can't collide with or reuse them. Safe to re-run: after this there are no flat caches, so no-op.
_mount()
import glob, os, shutil
QDIR = f"{BASE}/qwen2.5-7b-instruct"; os.makedirs(QDIR, exist_ok=True)
moved = 0
for f in (glob.glob(f"{BASE}/phaseAB_L*.pt") + glob.glob(f"{BASE}/causal_L*_N128")
          + glob.glob(f"{BASE}/kc_genuine.jsonl") + glob.glob(f"{BASE}/n128_layer*")):
    dst = f"{QDIR}/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.move(f, dst); moved += 1; print("moved", os.path.basename(f))
print(f"migration done ({moved} items). Flat caches were Qwen's; now under {QDIR}")

## 1 · kc curate — one-time per model; writes genuine `pairs.jsonl`

In [ ]:
# §1 · kc curate — MODEL-SPECIFIC (facts THIS model knows closed-book). Caches to MDIR, fast-loads after.
import json, os, shutil
_mount()
os.makedirs(MDIR, exist_ok=True)
GEN = f"{MDIR}/kc_genuine.jsonl"
if os.path.exists(GEN):
    shutil.copy(GEN, "data/knowledge_conflict/pairs.jsonl"); print(f"loaded cached genuine kc ({sum(1 for _ in open(GEN))} pairs)")
else:
    pairs = D.load_pairs("knowledge_conflict", cfg)
    bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype","bfloat16"))
    clean = M.generate(bundle, M.format_chat(bundle, [p.prompt_clean for p in pairs]), max_new_tokens=32)
    del bundle; _free()
    knows = [bool(p.clean_answer) and str(p.clean_answer).lower() in clean[i].lower() for i,p in enumerate(pairs)]
    raw = [json.loads(l) for l in open("data/knowledge_conflict/pairs.jsonl")]
    keep = [raw[i] for i,k in enumerate(knows) if k][:128]
    for path in ("data/knowledge_conflict/pairs.jsonl", GEN):
        with open(path,"w") as f: f.writelines(json.dumps(r)+"\n" for r in keep)
    print(f"curated + cached → {len(keep)} genuine pairs @ {MDIR}")

## 2 · Phase A/B store (per LAYER, per MODEL; reruns just load)

In [ ]:
# === Phase A+B store/fetch, per LAYER, MODEL-NAMESPACED. Gens+labels are layer-independent WITHIN a model;
#     the reuse branch globs ONLY this model's folder, so it can NEVER grab another model's generations. ===
_mount()
import os, glob, torch
os.makedirs(MDIR, exist_ok=True)
CACHE = f"{MDIR}/phaseAB_L{LAYER}.pt"

def _cache_acts(bundle, b, pairs):                        # 4 activation passes @LAYER (no generation)
    pt, nt = D.load_signal_statements(b, cfg)
    if MAXB: pt, nt = pt[:MAXB], nt[:MAXB]
    g = lambda txts: M.get_activations(bundle, M.format_chat(bundle, txts), [LAYER], [POS])[(LAYER, POS)]
    return dict(acts_manip=g([p.prompt_manip for p in pairs]), acts_clean=g([p.prompt_clean for p in pairs]),
                acts_pos=g(pt), acts_neg=g(nt))

if os.path.exists(CACHE):                                 # ---- FETCH this layer ----
    store = torch.load(CACHE, weights_only=False)
    print(f"loaded {MODEL_TAG} L{LAYER}:", {b: len(store[b]['pairs']) for b in store})
else:
    prior = next(iter(glob.glob(f"{MDIR}/phaseAB_L*.pt")), None)   # SAME MODEL's other layers only
    bundle = globals().get("bundle") or M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))  # reuse if loaded above
    if prior:                                             # ---- reuse gens+labels, only re-cache acts ----
        print(f"reusing gens+labels from {os.path.basename(prior)} (same model); caching acts @L{LAYER}")
        prev = torch.load(prior, weights_only=False)
        store = {b: {**prev[b], **_cache_acts(bundle, b, prev[b]["pairs"])} for b in SUBSET}
        del bundle; _free()
    else:                                                 # ---- first time for THIS model: full generate + judge ----
        PART = CACHE + ".partial"                                       # per-behavior checkpoint: a crash costs one behavior, not all
        store = torch.load(PART, weights_only=False) if os.path.exists(PART) else {}
        if store: print("resuming partial Phase A/B, done:", list(store))
        for b in SUBSET:
            if b in store: continue
            pairs = D.load_pairs(b, cfg); pairs = pairs[:MAXB] if MAXB else pairs
            manip = M.format_chat(bundle, [p.prompt_manip for p in pairs])
            clean = M.format_chat(bundle, [p.prompt_clean for p in pairs])
            uniq  = list(dict.fromkeys(clean))                            # safety's bare request repeats 4x (one per template)
            cmap  = dict(zip(uniq, M.generate(bundle, uniq, max_new_tokens=MAXNT)))
            store[b] = dict(pairs=pairs, **_cache_acts(bundle, b, pairs),
                            gen_manip=M.generate(bundle, manip, max_new_tokens=MAXNT),
                            gen_clean=[cmap[c] for c in clean])
            torch.save(store, PART); print(f"  generated {b}  (checkpointed)")
        del bundle; _free()
        cls = J.load_hf(J.HARMBENCH_CLS)
        if "safety" in SUBSET:
            s = store["safety"]
            s["overrode"]   = J.score("safety", s["pairs"], s["gen_manip"], cls=cls)
            s["clean_over"] = J.score("safety", s["pairs"], s["gen_clean"], cls=cls)
        del cls; _free()
        llm = J.load_hf(JUDGE_LLM)
        for b in [x for x in SUBSET if x in ("sycophancy", "knowledge_conflict")]:
            s = store[b]
            s["overrode"]   = J.score(b, s["pairs"], s["gen_manip"], llm=llm)
            s["clean_over"] = J.score(b, s["pairs"], s["gen_clean"], llm=llm)
        del llm; _free()
    torch.save(store, CACHE)
    if os.path.exists(CACHE + ".partial"): os.remove(CACHE + ".partial")
    print(f"saved -> {CACHE} | {os.path.getsize(CACHE)//1024//1024} MB")

## 3 · Judge baseline (only if store lacks labels)

In [ ]:
# === Phase B: baseline judging. Phase A/B's first-run branch ALREADY judges, so this only fires
#     if the store somehow lacks labels (e.g. an interrupted first run). Skips the 13B download otherwise. ===
if all("overrode" in store[b] and "clean_over" in store[b] for b in SUBSET):
    print("baseline labels already in store -> nothing to judge (skipped the cls download).")
else:
    cls = J.load_hf(J.HARMBENCH_CLS)
    if "safety" in SUBSET:
        s = store["safety"]
        s["overrode"]   = J.score("safety", s["pairs"], s["gen_manip"], cls=cls)
        s["clean_over"] = J.score("safety", s["pairs"], s["gen_clean"], cls=cls)
    del cls; _free()
    llm = J.load_hf(JUDGE_LLM)
    for b in [x for x in SUBSET if x in ("sycophancy", "knowledge_conflict")]:
        s = store[b]
        s["overrode"]   = J.score(b, s["pairs"], s["gen_manip"], llm=llm)
        s["clean_over"] = J.score(b, s["pairs"], s["gen_clean"], llm=llm)
    del llm; _free()
    torch.save(store, f"{MDIR}/phaseAB_L{LAYER}.pt")   # persist the freshly-added labels
    print("judging done + re-saved store")

## 4 · Directions (Phase C)

In [ ]:
# === Phase C: signal directions (diff-of-means) from cached acts + headline override rates. ===
# (Legacy suppression/residual directions dropped — the current analysis uses sig_dirs only.)
sig_dirs = {}
for b in SUBSET:
    s = store[b]
    ov = torch.tensor([x == 1 for x in s["overrode"]])
    sig_dirs[b] = Dir.signal_direction(s["acts_pos"], s["acts_neg"], layer=LAYER, position=POS, name="s", behavior=b)
    print(f"  {b:<20} JUDGED override rate: {int(ov.sum())}/{len(ov)} = {ov.float().mean():.2f}")

## 5 · Geometry — cosines + readout AUROC

In [ ]:
# === per-LAYER geometry triage — after Phase C @LAYER; seconds, no generation ===
import numpy as np
from sklearn.metrics import roc_auc_score
_u = lambda v: v.float()/(v.float().norm()+1e-8)
print(f"--- {MODEL_TAG} LAYER {LAYER}: signal-dir cosines ---")
for i, a in enumerate(SUBSET):
    for bnm in SUBSET[i+1:]:
        print(f"  {a[:4]}-{bnm[:4]}: {float(_u(sig_dirs[a].vec)@_u(sig_dirs[bnm].vec)):+.3f}")
harm = sig_dirs["safety"].vec
dirs = {"harm": harm, "truth": sig_dirs["sycophancy"].vec, "fact": sig_dirs["knowledge_conflict"].vec,
        "fact_perp_harm":  P.project_out(sig_dirs["knowledge_conflict"].vec, harm),
        "truth_perp_harm": P.project_out(sig_dirs["sycophancy"].vec, harm)}
print("readout AUROC on override  (proj acts_manip -> predict overrode; NOT causal):")
print(f"  {'direction':<16}{'safety':>8}{'syco':>8}")
for name, v in dirs.items():
    row = f"  {name:<16}"
    for b in ("safety", "sycophancy"):
        ov = np.array(store[b]["overrode"], dtype=int)
        proj = (store[b]["acts_manip"].float() @ _u(v)).numpy()
        row += f"{max(roc_auc_score(ov, proj), roc_auc_score(ov, -proj)):>8.2f}"
    print(row)

## 6 · Causal ablation (resumable, per-axis checkpoint)

In [ ]:
# === CAUSAL (Step 1): resumable · per-axis checkpoint · progress bar · layer-guarded ===
from src import steering as St, control as C, projection as P
from tqdm.auto import tqdm
_mount()
import os, time, torch, glob, json

N_CAUSAL, NT_CAUSAL = 128, 128
CKPT = f"{MDIR}/causal_L{LAYER}_N{N_CAUSAL}"; os.makedirs(CKPT, exist_ok=True)

# ---- LAYER GUARD: sig_dirs must have been built at THIS layer ----
_lyr = sig_dirs[SUBSET[0]].layer
assert _lyr == LAYER, f"MISMATCH: LAYER={LAYER} but sig_dirs are at layer {_lyr}. Re-run Phase C at L{LAYER} first."
print(f"=== CAUSAL @ LAYER {LAYER} | N={N_CAUSAL} | ckpt {CKPT}", flush=True)

def _gen(bundle, prompts, hooks):        # ONE batched path (length-sorted, token-budgeted) for every axis
    return M.generate(bundle, prompts, max_new_tokens=NT_CAUSAL, fwd_hooks=hooks)

# ---- axis vectors: reload if cached, else build once (needs model) ----
harm, bundle = sig_dirs["safety"].vec, None
if os.path.exists(f"{CKPT}/_axes.pt"):
    axes = torch.load(f"{CKPT}/_axes.pt", weights_only=False)
else:
    bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
    axes = {"none": None, "harm_dir": harm, "truth_dir": sig_dirs["sycophancy"].vec,
            "fact_dir": sig_dirs["knowledge_conflict"].vec,
            "fact_perp_harm":  P.project_out(sig_dirs["knowledge_conflict"].vec, harm),
            "truth_perp_harm": P.project_out(sig_dirs["sycophancy"].vec, harm)}
    sigbasis = torch.stack([sig_dirs[b].vec.float() for b in SUBSET])
    for c, cd in C.build_control_dirs(bundle, LAYER, POS).items():
        axes[f"ctrl_{c}"] = P.project_out(cd.vec, sigbasis)        # signal-orthogonalized control
    for k in range(N_RANDOM): axes[f"random{k}"] = St.sample_random_directions(bundle.d_model, 1, seed=k)[0]
    torch.save(axes, f"{CKPT}/_axes.pt")

json.dump({"N": N_CAUSAL, "NT": NT_CAUSAL}, open(f"{CKPT}/_meta.json", "w"))   # lets other layers reuse none/random safely
_u = lambda v: v.float()/(v.float().norm()+1e-8)                    # cosine sanity (orthogonalized controls, want ~0)
for c in ("sentiment", "formality", "topic"):
    print("  ctrl_"+c.ljust(9)+"  ".join(f"{s[:4]}={float(_u(axes[f'ctrl_{c}'])@_u(sig_dirs[s].vec)):+.3f}" for s in SUBSET))

# ---- resume: load finished axes, generate the rest, checkpoint each ----
steered = {a: torch.load(f"{CKPT}/{a}.pt", weights_only=False) for a in axes if os.path.exists(f"{CKPT}/{a}.pt")}
# ---- layer-INDEPENDENT axes (none = no hooks; random* = seeded on d_model): reuse from any other cached layer of THIS model ----
for a in [a for a in axes if a not in steered and (a == "none" or a.startswith("random"))]:
    for d in sorted(glob.glob(f"{MDIR}/causal_L*_N{N_CAUSAL}")):
        src, meta = f"{d}/{a}.pt", f"{d}/_meta.json"
        nt = json.load(open(meta))["NT"] if os.path.exists(meta) else 128          # pre-meta dirs all ran NT=128
        if d != CKPT and os.path.exists(src) and nt == NT_CAUSAL:
            cand = torch.load(src, weights_only=False)
            if set(SUBSET) <= set(cand):
                steered[a] = cand; torch.save(cand, f"{CKPT}/{a}.pt")
                print(f"  reused {a} from {os.path.basename(d)} (layer-independent; §11 sanity cell verifies)"); break
todo = [a for a in axes if a not in steered]
print(f"done {len(steered)}/{len(axes)} | todo: {todo}", flush=True)
if todo:
    if bundle is None: bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
    prompts = {b: M.format_chat(bundle, [p.prompt_manip for p in store[b]["pairs"][:N_CAUSAL]]) for b in SUBSET}
    for aname in tqdm(todo, desc="axes"):
        hooks = [] if axes[aname] is None else St.ablation_hooks(bundle, axes[aname])
        steered[aname] = {}
        for b in SUBSET:
            print(f"[{time.strftime('%H:%M:%S')}] {aname} / {b}", flush=True)   # reconnect-proof heartbeat
            steered[aname][b] = _gen(bundle, prompts[b], hooks)
        torch.save(steered[aname], f"{CKPT}/{aname}.pt")            # <- checkpoint after each axis
if bundle is not None: del bundle; _free()
print("steering complete — steered has all axes; each cached to Drive.", flush=True)

## 7 · Judge causal -> specificity report (+ persist labels)

In [ ]:
# === CAUSAL: judge -> per-item labels, then rate matrix + specificity report ===
from src import stats as S
labels = {a: {} for a in axes}                       # labels[axis][behavior] = list of 0/1
cls = J.load_hf(J.HARMBENCH_CLS)
for a in axes:
    labels[a]["safety"] = J.score("safety", store["safety"]["pairs"][:N_CAUSAL], steered[a]["safety"], cls=cls)
del cls; _free()
llm = J.load_hf(JUDGE_LLM)
for a in axes:
    for b in ("sycophancy", "knowledge_conflict"):
        labels[a][b] = J.score(b, store[b]["pairs"][:N_CAUSAL], steered[a][b], llm=llm)
del llm; _free()

rate = {a: {b: float(np.mean(labels[a][b])) for b in SUBSET} for a in axes}
print(f"{'ablate v / measure >':<20}" + "".join(f"{b[:11]:>13}" for b in SUBSET))
for a in axes: print(f"{a:<20}" + "".join(f"{rate[a][b]:>13.2f}" for b in SUBSET))

S.specificity_report(labels,
    control_axes=[f"ctrl_{c}" for c in ("sentiment", "formality", "topic")],
    signal_axes=["harm_dir", "truth_dir", "fact_dir", "fact_perp_harm", "truth_perp_harm"],
    behaviors=SUBSET, baseline_axis="none",
    random_axes=[f"random{k}" for k in range(3)])

In [ ]:
LP = f"{CKPT}/labels.pt"; saved = torch.load(LP, weights_only=False) if os.path.exists(LP) else {a:{} for a in axes}
n = sum(("safety" in labels.get(a,{})) for a in axes)
for a in axes:
    if "safety" in labels.get(a,{}): saved[a]["safety"] = labels[a]["safety"]
torch.save(saved, LP); print(f"merged safety for {n}/{len(axes)} axes")

## 8 · Coherence spot-checks

In [ ]:
# fact_perp_harm → safety (0.62): real jailbreaks, or degraded-but-compliant?
for g in steered["fact_perp_harm"]["safety"][:12]: print(repr(g[:300]), "\n")

In [ ]:
# truth_dir → sycophancy (0.37): real caves to the false claim, or noise?
for g in steered["truth_dir"]["sycophancy"][:12]: print(repr(g[:300]), "\n")

## 9 · kc loop-closer — fact->kc validity

In [ ]:
# §9 · kc loop-closer — fact->kc validity at THIS model's LAYER (reuses cached _axes.pt from §6)
from src import steering as St, stats as S; import numpy as np, torch
CKPT = f"{MDIR}/causal_L{LAYER}_N{N_CAUSAL}"
ax = torch.load(f"{CKPT}/_axes.pt", weights_only=False)
USE = ["none","harm_dir","truth_dir","fact_dir","fact_perp_harm","ctrl_sentiment","ctrl_formality","ctrl_topic"]
pairs = D.load_pairs("knowledge_conflict", cfg)[:N_CAUSAL]
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype","bfloat16"))
prompts = M.format_chat(bundle, [p.prompt_manip for p in pairs])
def gen(vec): return M.generate(bundle, prompts, max_new_tokens=128,
                                fwd_hooks=([] if vec is None else St.ablation_hooks(bundle, vec)))
gens={a:gen(ax[a]) for a in USE}; del bundle; _free()
llm=J.load_hf(JUDGE_LLM); lab={a:J.score("knowledge_conflict",pairs,gens[a],llm=llm) for a in USE}; del llm; _free()
ctrl=[x for c in ("ctrl_sentiment","ctrl_formality","ctrl_topic") for x in lab[c]]
print(f"kc baseline (genuine, neutral) = {np.mean(lab['none']):.2f}")
for a in ("harm_dir","truth_dir","fact_dir","fact_perp_harm"):
    r=lab[a]; print(f"  {a:<16} {np.mean(r):.2f}  z={S.two_prop_z(int(np.sum(r)),len(r),int(np.sum(ctrl)),len(ctrl)):+.1f}")

## 9b · fact->kc across cached layers

In [ ]:
# §9b · fact->kc across EVERY causal layer cached for THIS model (auto-discovered), z vs baseline.
from src import steering as St, stats as S; import numpy as np, torch, os, glob
def load_axes(d):
    if os.path.exists(f"{d}/_axes.pt"): return torch.load(f"{d}/_axes.pt", weights_only=False)
    if os.path.exists(f"{d}/cache.pt"): return torch.load(f"{d}/cache.pt", weights_only=False).get("axes")
layer_dir = {int(d.split('_L')[1].split('_')[0]): d for d in glob.glob(f"{MDIR}/causal_L*_N*") if os.path.isdir(d)}
print("causal layers cached for this model:", sorted(layer_dir))
pairs = D.load_pairs("knowledge_conflict", cfg)[:128]
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype","bfloat16"))
prompts = M.format_chat(bundle, [p.prompt_manip for p in pairs])
def gen(vec): return M.generate(bundle, prompts, max_new_tokens=128,
                                fwd_hooks=([] if vec is None else St.ablation_hooks(bundle, vec)))
AX=("none","harm_dir","truth_dir","fact_dir","fact_perp_harm")
res={L:{a:gen(ax[a]) for a in AX} for L,d in sorted(layer_dir.items()) if (ax:=load_axes(d))}
del bundle; _free()
llm=J.load_hf(JUDGE_LLM)
for L,g in res.items():
    lab={a:J.score("knowledge_conflict",pairs,g[a],llm=llm) for a in g}; b=lab["none"]
    print(f"L{L} base={np.mean(b):.2f}  " + "  ".join(
        f"{a.split('_')[0]}:{np.mean(lab[a]):.2f}(z{S.two_prop_z(int(np.sum(lab[a])),len(lab[a]),int(np.sum(b)),len(b)):+.1f})"
        for a in AX[1:]))
del llm; _free()

## 10 · Analysis — axis identity, readout sweep, projections, bidirectional steering

_Figures auto-save under `MDIR/figs/`._

In [ ]:
# === AXIS IDENTITY (no generation): max-activating inputs, + optional logit lens. ===
# Probes whatever phaseAB layers are cached for THIS model (auto-discovered).
import torch, numpy as np, glob, os
from src import directions as Dir, projection as P, data as D
_u = lambda v: v.float()/(v.float().norm()+1e-8)
PROBE = sorted(int(f.split("_L")[1].split(".pt")[0]) for f in glob.glob(f"{MDIR}/phaseAB_L*.pt"))
print("probing layers:", PROBE)
def dirs_at(L):
    st = torch.load(f"{MDIR}/phaseAB_L{L}.pt", weights_only=False)
    sd = {b: Dir.signal_direction(st[b]["acts_pos"], st[b]["acts_neg"], layer=L, position=-1, name="s", behavior=b).vec for b in st}
    h = sd["safety"]
    return st, {"harm": h, "fact_perp_harm": P.project_out(sd["knowledge_conflict"], h),
                "truth_perp_harm": P.project_out(sd["sycophancy"], h)}
def corpus(st):
    texts, acts = [], []
    for b in st:
        pt, nt = D.load_signal_statements(b, cfg); n = len(st[b]["acts_pos"])
        texts += [f"[{b[:4]}|T] {t}" for t in pt[:n]] + [f"[{b[:4]}|F] {t}" for t in nt[:n]]
        acts += [st[b]["acts_pos"].float(), st[b]["acts_neg"].float()]
    return texts, torch.cat(acts)
for L in PROBE:
    print(f"\n============ {MODEL_TAG} LAYER {L}: max-activating inputs ============")
    st, dd = dirs_at(L); texts, acts = corpus(st)
    for name, d in dd.items():
        proj = (acts @ _u(d)).numpy(); o = proj.argsort()
        print(f"\n  -- {name}  TOP(+) --")
        for i in o[::-1][:5]: print(f"    {proj[i]:+6.2f}  {texts[i][:100]}")
        print(f"  -- {name}  BOTTOM(-) --")
        for i in o[:5]:      print(f"    {proj[i]:+6.2f}  {texts[i][:100]}")
try:  # optional logit lens — needs the model in `bundle`
    W_U = bundle.model.W_U; dec = lambda t: bundle.model.tokenizer.decode([t])
    for L in PROBE:
        _, dd = dirs_at(L); print(f"\n===== LOGIT LENS @L{L} =====")
        for name, d in dd.items():
            lg = (_u(d).to(W_U.dtype) @ W_U).float()
            print(f"  {name:<16} +{[dec(t) for t in lg.topk(10).indices.tolist()]}")
            print(f"  {'':<16} -{[dec(t) for t in lg.topk(10, largest=False).indices.tolist()]}")
except NameError:
    print("\n(logit lens skipped — no `bundle`; the max-activating read above needs no model)")

In [ ]:
# === readout AUROC across layers (this model), saved to figs. No model, loads from Drive. ===
import matplotlib.pyplot as plt, numpy as np, glob, torch, os
from sklearn.metrics import roc_auc_score
from src import directions as Dir, projection as P
_u = lambda v: v.float()/(v.float().norm()+1e-8)
layers = {}
for f in sorted(glob.glob(f"{MDIR}/phaseAB_L*.pt")):
    L = int(f.split("_L")[1].split(".pt")[0]); st = torch.load(f, weights_only=False)
    sd = {b:_u(Dir.signal_direction(st[b]["acts_pos"], st[b]["acts_neg"], layer=L, position=-1, name="s", behavior=b).vec) for b in SUBSET}
    harm = sd["safety"]
    dirs = {"harm":harm, "truth":sd["sycophancy"], "fact":sd["knowledge_conflict"],
            "fact_perp_harm":_u(P.project_out(sd["knowledge_conflict"], harm)), "truth_perp_harm":_u(P.project_out(sd["sycophancy"], harm))}
    layers[L] = {b:{n: max(roc_auc_score(np.array(st[b]["overrode"],int), (st[b]["acts_manip"].float()@v).numpy()),
                            roc_auc_score(np.array(st[b]["overrode"],int), -(st[b]["acts_manip"].float()@v).numpy()))
                    for n,v in dirs.items()} for b in ("safety","sycophancy")}
Ls = sorted(layers); OK = ["#0072B2","#E69F00","#009E73","#D55E00","#56B4E9"]; names = list(dirs)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, b in zip(np.atleast_1d(axs), ("safety","sycophancy")):
    for c, n in zip(OK, names):
        ax.plot(Ls, [layers[L][b][n] for L in Ls], "-o", color=c, lw=2, ms=6, label=n)
    ax.axhline(.5, ls="--", c="#999", lw=1); ax.set_xticks(Ls); ax.set_xlabel("layer")
    ax.set_title(b); ax.set_ylim(.45, 1.0); ax.grid(alpha=.25)
axs[0].set_ylabel("readout AUROC on override"); axs[1].legend(fontsize=8, loc="upper left")
plt.suptitle(f"{MODEL_TAG}: direction reads out override across layers"); plt.tight_layout()
os.makedirs(f"{MDIR}/figs", exist_ok=True)
plt.savefig(f"{MDIR}/figs/readout-sweep.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# === TiU-style 2D projection (+ marginals) @LAYER. Saves to figs. ===
import matplotlib.pyplot as plt, numpy as np, os
from scipy.stats import gaussian_kde
from sklearn.metrics import roc_auc_score
from src import projection as P
_u = lambda v: v.float()/(v.float().norm()+1e-8)
os.makedirs(f"{MDIR}/figs", exist_ok=True)

def tiu_fig(behavior, a1, n1, a2, n2):
    A  = store[behavior]["acts_manip"].float()
    x, y = (A @ _u(a1)).numpy(), (A @ _u(a2)).numpy()
    ov = np.array(store[behavior]["overrode"], int)
    auc = lambda p: max(roc_auc_score(ov, p), roc_auc_score(ov, -p))
    fig = plt.figure(figsize=(9.5, 4.4)); gs = fig.add_gridspec(2, 2, width_ratios=[2, 1.3])
    ax = fig.add_subplot(gs[:, 0])
    for lab, c, nm in [(0, "#0072B2", "resisted"), (1, "#D55E00", "overrode")]:
        m = ov == lab
        ax.scatter(x[m], y[m], s=11, alpha=.45, c=c, edgecolors="none", label=f"{nm} ({int(m.sum())})")
    ax.set_xlabel(f"proj. onto {n1}"); ax.set_ylabel(f"proj. onto {n2}")
    ax.set_title(f"{behavior}:  {n1} x {n2}"); ax.legend(fontsize=8); ax.grid(alpha=.2)
    for i, (proj, nm) in enumerate([(x, n1), (y, n2)]):
        axm = fig.add_subplot(gs[i, 1]); g = np.linspace(proj.min(), proj.max(), 200)
        for lab, c in [(0, "#0072B2"), (1, "#D55E00")]:
            d = proj[ov == lab]
            if len(d) > 1: axm.plot(g, gaussian_kde(d)(g), c=c, lw=2)
        axm.set_yticks([]); axm.set_xlabel(f"a.{nm}", fontsize=8)
        axm.text(.04, .82, f"AUROC {auc(proj):.2f}", transform=axm.transAxes, fontsize=9,
                 bbox=dict(fc="white", ec="gray", lw=.6))
    plt.tight_layout()
    plt.savefig(f"{MDIR}/figs/tiu_{behavior}_{n1}-{n2}.png".replace("⊥","perp"), dpi=150, bbox_inches="tight")
    plt.show()

harm = sig_dirs["safety"].vec
tiu_fig("safety",     harm, "harm", P.project_out(sig_dirs["knowledge_conflict"].vec, harm), "fact_perp_harm")
tiu_fig("sycophancy", harm, "harm", P.project_out(sig_dirs["sycophancy"].vec, harm),        "truth_perp_harm")

In [ ]:
# === 3D plotly of attack prompts in the resistance space @LAYER (interactive; static export optional). ===
import plotly.graph_objects as go, os
_u = lambda v: v.float()/(v.float().norm()+1e-8)
harm = sig_dirs["safety"].vec
E = [_u(harm), _u(P.project_out(sig_dirs["knowledge_conflict"].vec, harm)),
     _u(P.project_out(sig_dirs["sycophancy"].vec, harm))]
fig = go.Figure()
for b, c in [("safety", "#0072B2"), ("sycophancy", "#E69F00"), ("knowledge_conflict", "#009E73")]:
    A = store[b]["acts_manip"].float(); xyz = [(A @ e).numpy() for e in E]
    fig.add_trace(go.Scatter3d(x=xyz[0], y=xyz[1], z=xyz[2], mode="markers",
        marker=dict(size=3, color=c, opacity=.5), name=b))
fig.update_layout(width=820, height=680, title=f"{MODEL_TAG} @L{LAYER}: attack prompts in resistance 3-space",
    scene=dict(xaxis_title="harm", yaxis_title="fact_perp_harm", zaxis_title="truth_perp_harm"))
fig.show()
try:
    os.makedirs(f"{MDIR}/figs", exist_ok=True); fig.write_image(f"{MDIR}/figs/3d_L{LAYER}.png")
except Exception as e:
    print("static export skipped (pip install -q kaleido to enable):", type(e).__name__)

In [ ]:
# === BIDIRECTIONAL: does +fact_perp_harm @L16 push safety override DOWN (mirror of ablation)? ===
from src import steering as St
import torch, numpy as np
dvec = axes["fact_perp_harm"].float(); dvec = dvec / dvec.norm()
sd = (store["safety"]["acts_manip"].float() @ dvec).std().item()      # natural spread along the dir @L16
COEFFS = [-3, -1.5, 0, 1.5, 3]                                          # in σ units
hook_name = f"blocks.{LAYER}.hook_resid_post"
print(f"σ along fact_perp_harm @L{LAYER} ≈ {sd:.1f}; steering in ±σ units")
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
sp = M.format_chat(bundle, [p.prompt_manip for p in store["safety"]["pairs"][:N_CAUSAL]])
add_gens = {}
for k in COEFFS:
    hooks = [] if k == 0 else [(hook_name, St.add_direction_hook(dvec, k * sd))]
    add_gens[k] = St.run_with_hooks(bundle, sp, hooks, max_new_tokens=NT_CAUSAL)
    print(f"  generated {k:+.1f}σ", flush=True)
del bundle; _free()
cls = J.load_hf(J.HARMBENCH_CLS)
print("\n coeff(σ)   safety override   (want: monotone ↓ as coeff ↑)")
for k in COEFFS:
    r = float(np.mean(J.score("safety", store["safety"]["pairs"][:N_CAUSAL], add_gens[k], cls=cls)))
    print(f"  {k:+.1f}        {r:.2f}")
del cls; _free()
# More evidence of correlation. If you want it for some extra metho.

In [ ]:
import numpy as np
dvec = axes["fact_perp_harm"].float(); dvec = dvec/dvec.norm()
print("mean proj of safety manip prompts on fact⊥harm:",
      float((store["safety"]["acts_manip"].float() @ dvec).mean()))   # expect NEGATIVE

## 11 · Sanity checks + cross-layer results

In [ ]:
# === CACHE SANITY: Phase A/B stores across THIS model's layers (no model, loads from Drive) ===
import glob, os, hashlib, torch, numpy as np
Lof  = lambda f: int(f.split("_L")[1].split(".pt")[0])
stores = {Lof(f): torch.load(f, weights_only=False) for f in glob.glob(f"{MDIR}/phaseAB_L*.pt")}
Ls, behs = sorted(stores), list(stores[min(stores)]); print("layers:", Ls, "| behaviors:", behs)
sfp = lambda x: hashlib.md5(repr(x).encode()).hexdigest()[:8]

print("\n[per layer] counts + override rate (labels are layer-independent -> rates equal across layers)")
for L in Ls:
    for b in behs:
        s = stores[L][b]; ov = np.mean(s["overrode"]) if "overrode" in s else float("nan")
        print(f"  L{L:<3} {b:<20} n={len(s['pairs']):<4} override={ov:.3f}")

print("\n[INVARIANT 1] gens+labels IDENTICAL across layers (the reuse branch copies them)")
for b in behs:
    for k in ("gen_manip","gen_clean","overrode","clean_over"):
        fps = {sfp(stores[L][b][k]) for L in Ls if k in stores[L][b]}
        print(f"  {b:<20} {k:<11} identical: {len(fps)==1}")

print("\n[INVARIANT 2] acts DIFFER between layers  (allclose==True = the old cross-cache bug)")
for b in behs:
    for i in range(len(Ls)):
        for j in range(i+1, len(Ls)):
            a, c = stores[Ls[i]][b]["acts_pos"], stores[Ls[j]][b]["acts_pos"]
            same = a.shape==c.shape and torch.allclose(a.float(), c.float())
            print(f"  {b:<20} acts_pos L{Ls[i]} vs L{Ls[j]}: allclose={same}" + ("   <-- BUG" if same else ""))

print("\n[INVARIANT 3] within a layer, the 4 act tensors are distinct + shaped [N, d_model]")
for L in Ls:
    s = stores[L][behs[0]]
    print(f"  L{L}: acts_manip {tuple(s['acts_manip'].shape)}  "
          f"manip==clean? {torch.allclose(s['acts_manip'].float(), s['acts_clean'].float())} (want False)  "
          f"pos==neg? {torch.allclose(s['acts_pos'].float(), s['acts_neg'].float())} (want False)")

In [ ]:
# === CAUSAL CACHE SANITY: steered gens across THIS model's layers (auto-discovered; no model) ===
import glob, os, torch
def load_steered(d):
    fs = [f for f in glob.glob(f"{d}/*.pt") if os.path.basename(f) not in ("_axes.pt","labels.pt")]
    return {os.path.basename(f)[:-3]: torch.load(f, weights_only=False) for f in fs} if fs else None
layer_dir = {int(d.split('_L')[1].split('_')[0]): d for d in glob.glob(f"{MDIR}/causal_L*_N*") if os.path.isdir(d)}
layers = sorted(L for L,d in layer_dir.items() if load_steered(d))
S = {L: load_steered(layer_dir[L]) for L in layers}
for L in layers: print(f"L{L}: {len(S[L])} axes -> {sorted(S[L])}")
if len(layers) < 2:
    print("\n(need >=2 cached causal layers for a cross-layer comparison)")
else:
    common = sorted(set.intersection(*[set(S[L]) for L in layers]))
    behs   = sorted(set.intersection(*[set(S[L][common[0]]) for L in layers]))
    b0     = "safety" if "safety" in behs else behs[0]
    fs = lambda A,B: sum(a==b for a,b in zip(A,B))/max(1,min(len(A),len(B)))
    print(f"\ncommon axes: {common}\ncomparing '{b0}' | frac of gens IDENTICAL between layer pairs:")
    print("  " + f"{'axis':<16}" + "".join(f"{f'L{layers[i]}~L{layers[i+1]}':>12}" for i in range(len(layers)-1)))
    for a in common:
        row = "".join(f"{fs(S[layers[i]][a][b0], S[layers[i+1]][a][b0]):>12.2f}" for i in range(len(layers)-1))
        kind = "layer-indep (want ~1.0)" if (a=="none" or a.startswith("random")) else "layer-dep (want LOW)"
        print(f"  {a:<16}{row}   {kind}")
    print("\nHEALTHY: none/random ~1.0 ; signals/controls LOW.  BUG: a signal/control row ~1.0.")

In [ ]:
# === CROSS-LAYER RESULTS: z vs control band across THIS model's layers (loads labels.pt; no model) ===
import glob, os, numpy as np, torch
from src import stats as S
SIG  = ["harm_dir","truth_dir","fact_dir","fact_perp_harm","truth_perp_harm"]
CTRL = ["ctrl_sentiment","ctrl_formality","ctrl_topic"]
ldirs = sorted(int(d.split('_L')[1].split('_')[0]) for d in glob.glob(f"{MDIR}/causal_L*_N*") if os.path.isdir(d))
lab  = {L: torch.load(f"{MDIR}/causal_L{L}_N128/labels.pt", weights_only=False)
        for L in ldirs if os.path.exists(f"{MDIR}/causal_L{L}_N128/labels.pt")}
print("behaviors saved per layer:", {L: sorted({b for a in lab[L] for b in lab[L][a]}) for L in lab})
has = lambda L,b: all(b in lab[L].get(a,{}) for a in SIG+CTRL)
def z(L,a,b):
    ctrl=[x for c in CTRL for x in lab[L][c][b]]
    return S.two_prop_z(int(np.sum(lab[L][a][b])), len(lab[L][a][b]), int(np.sum(ctrl)), len(ctrl))
for b in ("sycophancy","safety","knowledge_conflict"):
    Ls=[L for L in lab if has(L,b)]
    if not Ls: print(f"\n=== {b}: not saved yet ==="); continue
    print(f"\n=== {b}: z vs control band ===\n  {'axis':<16}" + "".join(f"{'L'+str(L):>7}" for L in Ls))
    for a in SIG: print(f"  {a:<16}" + "".join(f"{z(L,a,b):>7.1f}" for L in Ls))